## Importing the relevant packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.graphics.tsaplots as sgt
import statsmodels.tsa.stattools as sts
from statsmodels.tsa.arima.model import ARIMA
from scipy.stats.distributions import chi2 
from arch import arch_model
from math import sqrt
import seaborn as sns
sns.set()

## Importing the Data and Pre-processing 

In [3]:
raw_csv_data = pd.read_csv("Index2018.csv") 
df_comp=raw_csv_data.copy()
df_comp.date = pd.to_datetime(df_comp.date, dayfirst = True)
df_comp.set_index("date", inplace=True)
df_comp=df_comp.asfreq('b')
df_comp=df_comp.ffill()

In [4]:
df_comp['market_value']=df_comp.ftse

In [5]:
del df_comp['spx']
del df_comp['dax']
del df_comp['ftse']
del df_comp['nikkei']
size = int(len(df_comp)*0.8)
df, df_test = df_comp.iloc[:size], df_comp.iloc[size:]

In [6]:
import warnings
warnings.filterwarnings("ignore")

## The LLR Test

In [7]:
def LLR_test(mod_1, mod_2, DF = 1):
    L1 = mod_1.fit(start_ar_lags = 11).llf
    L2 = mod_2.fit(start_ar_lags = 11).llf
    LR = (2*(L2-L1))    
    p = chi2.sf(LR, DF).round(3)
    return p

## Creating Returns

In [8]:
df['returns'] = df.market_value.pct_change(1)*100

## The Simple GARCH Model

In [9]:
# What is new in GARCH model Generalized AutoRegressive Conditional Heteroskedastic Model
# In this we can specify if we want previous Squared error terms which helps our model to fit better according to previous 
# data
# We can put Any previously learned Mean models inside ARCH and get better results


`When we increaes q values -> We are adding past shocks information `                                                    
`When we increase p values -> We are adding past volatility information(volatility clustering)`                           
`But Increasing p values is redundant since one previous volatility info already has its past volatility info and we are adding it again, it makes equation redundant and it performs worse`

In [11]:
# Forming Simple GARCH(1,1) Model
model_garch_11 = arch_model(df.returns[1:], mean='Constant',vol='GARCH',p = 1, q = 1)
result_garch_11 = model_garch_11.fit()
print(result_garch_11.summary())

Iteration:      1,   Func. Count:      6,   Neg. LLF: 6579303469.390623
Iteration:      2,   Func. Count:     15,   Neg. LLF: 2701100877.2298183
Iteration:      3,   Func. Count:     23,   Neg. LLF: 7009.030632045198
Iteration:      4,   Func. Count:     29,   Neg. LLF: 7024.035835212278
Iteration:      5,   Func. Count:     35,   Neg. LLF: 7010.712887007633
Iteration:      6,   Func. Count:     41,   Neg. LLF: 6975.418108744094
Iteration:      7,   Func. Count:     47,   Neg. LLF: 7092.271338807877
Iteration:      8,   Func. Count:     53,   Neg. LLF: 6973.879266228052
Iteration:      9,   Func. Count:     59,   Neg. LLF: 6970.088048943886
Iteration:     10,   Func. Count:     64,   Neg. LLF: 6970.058478413694
Iteration:     11,   Func. Count:     69,   Neg. LLF: 6970.0583674757745
Iteration:     12,   Func. Count:     74,   Neg. LLF: 6970.058366189882
Iteration:     13,   Func. Count:     78,   Neg. LLF: 6970.058366189167
Optimization terminated successfully    (Exit mode 0)
        

`GARCH 1_1 proves to be a better fit than those ARCH(11) lag Model`                                                       
`shows that the past variance term helped a lot in fitting the data properly, q =1`

## Higher-Lag GARCH Models

In [ ]:
# Forming GARCH models with 2 past sq residuals terms
model_garch_21 = arch_model(df.returns[1:], mean='Constant', vol='GARCH',p=2,q=1)
result_garch_21 = model_garch_21.fit()
print(result_garch_21.summary())

Iteration:      1,   Func. Count:      7,   Neg. LLF: 159537792891988.94
Iteration:      2,   Func. Count:     17,   Neg. LLF: 1848307647.4642267
Iteration:      3,   Func. Count:     25,   Neg. LLF: 10354.116095029069
Iteration:      4,   Func. Count:     33,   Neg. LLF: 7005.361877757425
Iteration:      5,   Func. Count:     40,   Neg. LLF: 8793.711867692436
Iteration:      6,   Func. Count:     47,   Neg. LLF: 7019.706996857094
Iteration:      7,   Func. Count:     54,   Neg. LLF: 6973.161800677724
Iteration:      8,   Func. Count:     61,   Neg. LLF: 7010.720410376376
Iteration:      9,   Func. Count:     69,   Neg. LLF: 6967.937760895733
Iteration:     10,   Func. Count:     76,   Neg. LLF: 6967.73124749643
Iteration:     11,   Func. Count:     82,   Neg. LLF: 6967.731020076215
Iteration:     12,   Func. Count:     87,   Neg. LLF: 6967.7310200761385
Optimization terminated successfully    (Exit mode 0)
            Current function value: 6967.731020076215
            Iterations: 1

`We see that alpha 2 is insignificant and beta 1 p value is 0 `

In [13]:
# Forming GARCH models with 2 past volatility terms
model_garch_21 = arch_model(df.returns[1:], mean='Constant', vol='GARCH',p=1,q=2)
result_garch_21 = model_garch_21.fit()
print(result_garch_21.summary())

Iteration:      1,   Func. Count:      7,   Neg. LLF: 136466776694655.33
Iteration:      2,   Func. Count:     17,   Neg. LLF: 808459246.0256559
Iteration:      3,   Func. Count:     25,   Neg. LLF: 10137.459118305613
Iteration:      4,   Func. Count:     33,   Neg. LLF: 7008.430986114414
Iteration:      5,   Func. Count:     40,   Neg. LLF: 6974.173831538361
Iteration:      6,   Func. Count:     46,   Neg. LLF: 6971.511859365692
Iteration:      7,   Func. Count:     52,   Neg. LLF: 6970.616224326323
Iteration:      8,   Func. Count:     58,   Neg. LLF: 6973.892231638968
Iteration:      9,   Func. Count:     65,   Neg. LLF: 6970.063506713137
Iteration:     10,   Func. Count:     71,   Neg. LLF: 6970.058391826686
Iteration:     11,   Func. Count:     77,   Neg. LLF: 6970.058366757141
Iteration:     12,   Func. Count:     83,   Neg. LLF: 6970.05836622724
Optimization terminated successfully    (Exit mode 0)
            Current function value: 6970.05836622724
            Iterations: 12
 

`See that beta 2 becomes insignificant thus the model is Not A good fit`

`The Best Model is GARCH(1,1)`

In [14]:
# Let us Try GARCH With Mean being some model
model_garch_ar  = arch_model(df.returns[1:],mean='AR',vol='GARCH',p=1,q=1)
result_garch_ar = model_garch_ar.fit()
print(result_garch_ar.summary())

Iteration:      1,   Func. Count:      6,   Neg. LLF: 6579303469.390623
Iteration:      2,   Func. Count:     15,   Neg. LLF: 2701100877.2298183
Iteration:      3,   Func. Count:     23,   Neg. LLF: 7009.030632045198
Iteration:      4,   Func. Count:     29,   Neg. LLF: 7024.035835212278
Iteration:      5,   Func. Count:     35,   Neg. LLF: 7010.712887007633
Iteration:      6,   Func. Count:     41,   Neg. LLF: 6975.418108744094
Iteration:      7,   Func. Count:     47,   Neg. LLF: 7092.271338807877
Iteration:      8,   Func. Count:     53,   Neg. LLF: 6973.879266228052
Iteration:      9,   Func. Count:     59,   Neg. LLF: 6970.088048943886
Iteration:     10,   Func. Count:     64,   Neg. LLF: 6970.058478413694
Iteration:     11,   Func. Count:     69,   Neg. LLF: 6970.0583674757745
Iteration:     12,   Func. Count:     74,   Neg. LLF: 6970.058366189882
Iteration:     13,   Func. Count:     78,   Neg. LLF: 6970.058366189167
Optimization terminated successfully    (Exit mode 0)
        

`Comparing With GARCH(1,1) w/ const mean this Model performs poorer`